# EXPT NO: 8 — GENERATION OF SYNTHETIC IMAGES USING A GENERATIVE ADVERSARIAL NETWORK (GAN)

**Name:** NIVASRI T  
**Roll No:** 24BAD081  
**Date:** 

---
## AIM
To implement and train a Generative Adversarial Network (GAN) to generate synthetic images by training a Generator and a Discriminator in an adversarial learning framework.

## SOFTWARE REQUIREMENTS
Python, TensorFlow / Keras, Google Colab, GitHub, NumPy, Matplotlib, Pandas, Scikit-learn

In [ ]:
# ---------------------------------------------------------------
# Experiment 8 : GAN - Synthetic Image Generation (MNIST)
# NAME    : NIVASRI T
# ROLL NO : 24BAD081
# ---------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.decomposition import PCA

SEED = 81                       # last digits of roll no. 24BAD081
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Name    : NIVASRI T")
print("Roll No : 24BAD081")
print("TensorFlow version:", tf.__version__)
print("GPU available     :", len(tf.config.list_physical_devices('GPU')) > 0)

---
## PART A: DATASET PREPARATION
Load MNIST, display sample images, explore dimensions / pixel range, normalise and reshape, and prepare batches.

In [ ]:
# ---- Part A.1 : Load the MNIST dataset ----
# Roll No: 24BAD081 | Name: NIVASRI T
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

print("Training images shape :", x_train.shape)
print("Training labels shape :", y_train.shape)
print("Test images shape     :", x_test.shape)
print("Image dtype           :", x_train.dtype)

In [ ]:
# ---- Part A.2 : Display sample images ----
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i], cmap='gray')
    ax.set_title(f"Label: {y_train[i]}", fontsize=9)
    ax.axis('off')
plt.suptitle("Sample Real MNIST Images  |  24BAD081 - NIVASRI T", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ---- Part A.3 : Explore dimensions and pixel-value range ----
print("Single image dimensions :", x_train[0].shape)
print("Pixel min value         :", x_train.min())
print("Pixel max value         :", x_train.max())
print("Pixel mean / std        : %.2f / %.2f" % (x_train.mean(), x_train.std()))

# Class distribution
dist = pd.Series(y_train).value_counts().sort_index()
dist.index.name = "Digit"
print("\nClass distribution:")
print(dist.to_frame("Count").T.to_string())

plt.figure(figsize=(6, 3))
plt.hist(x_train[:1000].ravel(), bins=50, color='steelblue')
plt.title("Pixel Value Distribution (raw 0-255)  |  24BAD081")
plt.xlabel("Pixel value"); plt.ylabel("Frequency")
plt.show()

In [ ]:
# ---- Part A.4 : Normalise to [-1, 1] and reshape to (28, 28, 1) ----
# The Generator uses tanh in its last layer, so real images must lie in [-1, 1].
x_train_gan = x_train.reshape(-1, 28, 28, 1).astype("float32")
x_train_gan = (x_train_gan - 127.5) / 127.5

print("Reshaped training data shape :", x_train_gan.shape)
print("New pixel min / max          :", x_train_gan.min(), "/", x_train_gan.max())

# ---- Part A.5 : Prepare batches of real images ----
BUFFER_SIZE = x_train_gan.shape[0]
BATCH_SIZE  = 256

train_dataset = (tf.data.Dataset.from_tensor_slices(x_train_gan)
                 .shuffle(BUFFER_SIZE, seed=SEED)
                 .batch(BATCH_SIZE, drop_remainder=True)
                 .prefetch(tf.data.AUTOTUNE))

for batch in train_dataset.take(1):
    print("One batch of real images     :", batch.shape)
print("Batches per epoch            :", len(train_dataset))
print("Dataset prepared by 24BAD081 - NIVASRI T")

---
## PART B: IMPLEMENTING THE GENERATOR AND DISCRIMINATOR

In [ ]:
# ---- Part B.1 : Generator (noise vector -> 28x28x1 image) ----
# Roll No: 24BAD081 | Name: NIVASRI T
NOISE_DIM = 100

def build_generator():
    model = keras.Sequential(name="Generator_24BAD081")
    model.add(layers.Input(shape=(NOISE_DIM,)))
    model.add(layers.Dense(7 * 7 * 256, use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(0.2))
    model.add(layers.Reshape((7, 7, 256)))

    model.add(layers.Conv2DTranspose(128, 5, strides=1, padding='same', use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(0.2))                        # 7 x 7

    model.add(layers.Conv2DTranspose(64, 5, strides=2, padding='same', use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(0.2))                        # 14 x 14

    model.add(layers.Conv2DTranspose(1, 5, strides=2, padding='same',
                                     use_bias=False, activation='tanh'))   # 28 x 28
    return model

generator = build_generator()
generator.summary()

In [ ]:
# ---- Part B.2 : Discriminator (image -> real/fake logit) ----
def build_discriminator():
    model = keras.Sequential(name="Discriminator_24BAD081")
    model.add(layers.Input(shape=(28, 28, 1)))
    model.add(layers.Conv2D(64, 5, strides=2, padding='same'))
    model.add(layers.LeakyReLU(0.2))
    model.add(layers.Dropout(0.3))

    model.add(layers.Conv2D(128, 5, strides=2, padding='same'))
    model.add(layers.LeakyReLU(0.2))
    model.add(layers.Dropout(0.3))

    model.add(layers.Flatten(name="features"))
    model.add(layers.Dense(1))                  # logit (no sigmoid; loss uses from_logits=True)
    return model

discriminator = build_discriminator()
discriminator.summary()

In [ ]:
# ---- Part B.3 : Loss functions and optimisers ----
cross_entropy = keras.losses.BinaryCrossentropy(from_logits=True)

def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output),  real_output)   # real -> 1
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)   # fake -> 0
    return real_loss + fake_loss

def generator_loss(fake_output):
    # Generator wants the Discriminator to label its fakes as real (1)
    return cross_entropy(tf.ones_like(fake_output), fake_output)

generator_optimizer     = keras.optimizers.Adam(1e-4, beta_1=0.5)
discriminator_optimizer = keras.optimizers.Adam(1e-4, beta_1=0.5)
print("Loss: Binary Cross-Entropy | Optimiser: Adam(lr=1e-4, beta_1=0.5)")

In [ ]:
# ---- Part B.4 : Combine Generator + Discriminator to construct the GAN ----
# For the combined model the Discriminator is frozen, so only the Generator learns.
discriminator.trainable = False
gan_input  = keras.Input(shape=(NOISE_DIM,))
gan_output = discriminator(generator(gan_input))
gan = keras.Model(gan_input, gan_output, name="GAN_24BAD081")
discriminator.trainable = True        # re-enabled for the manual training loop below

print("Combined GAN model (noise -> Generator -> Discriminator -> real/fake)")
gan.summary()

# Quick sanity check: untrained generator output
noise = tf.random.normal([1, NOISE_DIM])
untrained = generator(noise, training=False)
print("\nUntrained Generator output shape :", untrained.shape)
print("Discriminator score (logit)      :", float(discriminator(untrained, training=False)[0][0]))
plt.imshow(untrained[0, :, :, 0], cmap='gray'); plt.axis('off')
plt.title("Untrained Generator output (pure noise)  |  24BAD081"); plt.show()

---
## PART C: GAN TRAINING AND PERFORMANCE MONITORING
The Discriminator and Generator are trained alternately in every step:
1. Discriminator sees a batch of **real** images and a batch of **generated** images and learns to separate them.
2. Generator produces new images and is updated so the Discriminator classifies them as **real**.

In [ ]:
# ---- Part C.1 : Training step (alternating updates) ----
# Roll No: 24BAD081 | Name: NIVASRI T
EPOCHS           = 30
NUM_EXAMPLES     = 16
SAVE_EVERY       = 5
seed_noise       = tf.random.normal([NUM_EXAMPLES, NOISE_DIM], seed=SEED)   # fixed noise to track progress

@tf.function
def train_step(real_images):
    noise = tf.random.normal([tf.shape(real_images)[0], NOISE_DIM])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        fake_images = generator(noise, training=True)

        real_output = discriminator(real_images, training=True)
        fake_output = discriminator(fake_images, training=True)

        g_loss = generator_loss(fake_output)
        d_loss = discriminator_loss(real_output, fake_output)

    g_grads = gen_tape.gradient(g_loss, generator.trainable_variables)
    d_grads = disc_tape.gradient(d_loss, discriminator.trainable_variables)

    generator_optimizer.apply_gradients(zip(g_grads, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(d_grads, discriminator.trainable_variables))

    # Discriminator accuracy on this batch (logit > 0 means "real")
    d_acc = (tf.reduce_mean(tf.cast(real_output > 0, tf.float32)) +
             tf.reduce_mean(tf.cast(fake_output < 0, tf.float32))) / 2.0
    return g_loss, d_loss, d_acc

def show_generated(model, noise, epoch, save=False):
    imgs = model(noise, training=False)
    fig = plt.figure(figsize=(6, 6))
    for i in range(imgs.shape[0]):
        plt.subplot(4, 4, i + 1)
        plt.imshow(imgs[i, :, :, 0] * 127.5 + 127.5, cmap='gray')
        plt.axis('off')
    plt.suptitle(f"Generated images - Epoch {epoch}  |  24BAD081", fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
# ---- Part C.2 : Train for EPOCHS, recording losses and showing samples ----
import time
history = {"g_loss": [], "d_loss": [], "d_acc": []}

show_generated(generator, seed_noise, epoch=0)      # before training

for epoch in range(1, EPOCHS + 1):
    start = time.time()
    g_l, d_l, d_a = [], [], []
    for image_batch in train_dataset:
        g, d, a = train_step(image_batch)
        g_l.append(g.numpy()); d_l.append(d.numpy()); d_a.append(a.numpy())

    history["g_loss"].append(np.mean(g_l))
    history["d_loss"].append(np.mean(d_l))
    history["d_acc"].append(np.mean(d_a))

    print(f"Epoch {epoch:02d}/{EPOCHS} | G loss: {history['g_loss'][-1]:.4f} | "
          f"D loss: {history['d_loss'][-1]:.4f} | D acc: {history['d_acc'][-1]*100:5.1f}% | "
          f"{time.time()-start:.1f}s")

    if epoch % SAVE_EVERY == 0 or epoch == 1:
        show_generated(generator, seed_noise, epoch)

print("\nTraining complete - 24BAD081 NIVASRI T")

In [ ]:
# ---- Part C.3 : Plot Generator and Discriminator losses ----
fig, ax = plt.subplots(1, 2, figsize=(13, 4))

ax[0].plot(history["g_loss"], label="Generator loss", color='tab:red', marker='o', ms=3)
ax[0].plot(history["d_loss"], label="Discriminator loss", color='tab:blue', marker='o', ms=3)
ax[0].set_title("GAN Losses per Epoch  |  24BAD081")
ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Loss"); ax[0].legend(); ax[0].grid(alpha=.3)

ax[1].plot(np.array(history["d_acc"]) * 100, color='tab:green', marker='o', ms=3)
ax[1].axhline(50, ls='--', color='gray', label="Ideal equilibrium (50%)")
ax[1].set_title("Discriminator Accuracy per Epoch  |  24BAD081")
ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("Accuracy (%)"); ax[1].legend(); ax[1].grid(alpha=.3)

plt.tight_layout(); plt.show()

pd.DataFrame(history).round(4).rename_axis("Epoch").set_index(
    pd.RangeIndex(1, len(history["g_loss"]) + 1, name="Epoch")).tail(10)

---
## PART D: FEATURE / LATENT-SPACE VISUALIZATION, SYNTHETIC IMAGE GENERATION AND EXPERIMENT MANAGEMENT

> **Note:** The lab sheet's Part D wording (LSTM movie reviews / Encoder) is carried over from another experiment.  
> For a GAN, the equivalent tasks are: feed **new noise vectors** to the trained Generator, inspect the **latent space**, extract **feature representations** of images with the trained Discriminator, and compare generated vs. real images. These are done below.

In [ ]:
# ---- Part D.1 : Generate NEW synthetic images from fresh random noise ----
# Roll No: 24BAD081 | Name: NIVASRI T
new_noise  = tf.random.normal([32, NOISE_DIM], seed=2024)
new_images = generator(new_noise, training=False).numpy()

fig, axes = plt.subplots(4, 8, figsize=(14, 7))
for img, ax in zip(new_images, axes.flat):
    ax.imshow(img[:, :, 0], cmap='gray', vmin=-1, vmax=1); ax.axis('off')
plt.suptitle("32 New Synthetic Digits from Trained Generator  |  24BAD081 - NIVASRI T", fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
# ---- Part D.2 : Compare generated images with original dataset samples ----
real_idx   = np.random.choice(len(x_train_gan), 10, replace=False)
real_imgs  = x_train_gan[real_idx]
fake_imgs  = generator(tf.random.normal([10, NOISE_DIM], seed=7), training=False).numpy()

fig, axes = plt.subplots(2, 10, figsize=(16, 3.8))
for i in range(10):
    axes[0, i].imshow(real_imgs[i, :, :, 0], cmap='gray', vmin=-1, vmax=1); axes[0, i].axis('off')
    axes[1, i].imshow(fake_imgs[i, :, :, 0], cmap='gray', vmin=-1, vmax=1); axes[1, i].axis('off')
axes[0, 0].set_title("REAL", loc='left', fontsize=12, color='green')
axes[1, 0].set_title("GENERATED", loc='left', fontsize=12, color='red')
plt.suptitle("Real vs Generated Images  |  24BAD081 - NIVASRI T", fontsize=13)
plt.tight_layout(); plt.show()

# Discriminator's verdict (sigmoid of logit = probability of "real")
p_real = tf.sigmoid(discriminator(real_imgs, training=False)).numpy().ravel()
p_fake = tf.sigmoid(discriminator(fake_imgs, training=False)).numpy().ravel()
print("Mean P(real) for REAL images      : %.3f" % p_real.mean())
print("Mean P(real) for GENERATED images : %.3f" % p_fake.mean())

In [ ]:
# ---- Part D.3 : Feature extraction + dimensionality reduction (PCA) ----
# Use the trained Discriminator's 'features' layer as a feature extractor.
feature_extractor = keras.Model(discriminator.inputs,
                                discriminator.get_layer("features").output)

n = 1000
real_sample = x_train_gan[np.random.choice(len(x_train_gan), n, replace=False)]
fake_sample = generator(tf.random.normal([n, NOISE_DIM]), training=False).numpy()

f_real = feature_extractor.predict(real_sample, verbose=0)
f_fake = feature_extractor.predict(fake_sample, verbose=0)
print("Feature vector size per image :", f_real.shape[1])

pca = PCA(n_components=2, random_state=SEED)
z = pca.fit_transform(np.vstack([f_real, f_fake]))

plt.figure(figsize=(7, 6))
plt.scatter(z[:n, 0],  z[:n, 1],  s=8, alpha=.5, c='green', label='Real')
plt.scatter(z[n:, 0],  z[n:, 1],  s=8, alpha=.5, c='red',   label='Generated')
plt.title("PCA of Discriminator Features: Real vs Generated  |  24BAD081")
plt.xlabel("PC 1"); plt.ylabel("PC 2"); plt.legend(); plt.grid(alpha=.3)
plt.show()
print("Explained variance (PC1, PC2): %.1f%%, %.1f%%" % tuple(pca.explained_variance_ratio_ * 100))

In [ ]:
# ---- Part D.4 : Latent-space exploration (interpolation between two noise vectors) ----
z1 = tf.random.normal([1, NOISE_DIM], seed=11)
z2 = tf.random.normal([1, NOISE_DIM], seed=22)
steps = 10
alphas = np.linspace(0, 1, steps)
interp = np.vstack([(1 - a) * z1.numpy() + a * z2.numpy() for a in alphas])
interp_imgs = generator(interp, training=False).numpy()

fig, axes = plt.subplots(1, steps, figsize=(16, 2.2))
for i, ax in enumerate(axes):
    ax.imshow(interp_imgs[i, :, :, 0], cmap='gray', vmin=-1, vmax=1)
    ax.set_title(f"a={alphas[i]:.1f}", fontsize=8); ax.axis('off')
plt.suptitle("Latent-Space Interpolation (smooth morphing between digits)  |  24BAD081", fontsize=12, y=1.08)
plt.show()

In [ ]:
# ---- Part D.5 : Diversity check - how many different digit shapes does the Generator produce? ----
# Train a small classifier on real MNIST and use it to label the generated images.
clf = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, 3, activation='relu'), layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation='relu'), layers.MaxPooling2D(),
    layers.Flatten(), layers.Dense(64, activation='relu'), layers.Dense(10)
])
clf.compile('adam', keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=['accuracy'])
clf.fit(x_train_gan, y_train, epochs=2, batch_size=128, verbose=0)
x_test_gan = (x_test.reshape(-1, 28, 28, 1).astype('float32') - 127.5) / 127.5
print("Helper classifier test accuracy on REAL images: %.2f%%" % (clf.evaluate(x_test_gan, y_test, verbose=0)[1] * 100))

gen_big   = generator(tf.random.normal([2000, NOISE_DIM]), training=False).numpy()
pred      = np.argmax(clf.predict(gen_big, verbose=0), axis=1)
conf      = tf.reduce_max(tf.nn.softmax(clf.predict(gen_big, verbose=0)), axis=1).numpy()

counts = pd.Series(pred).value_counts().sort_index()
plt.figure(figsize=(7, 3.5))
plt.bar(counts.index, counts.values, color='teal')
plt.xticks(range(10)); plt.xlabel("Predicted digit class"); plt.ylabel("Count (of 2000)")
plt.title("Class Diversity of Generated Images  |  24BAD081"); plt.show()
print("Average classifier confidence on generated images: %.1f%%" % (conf.mean() * 100))
print("Classes covered:", len(counts), "/ 10")

### Observations (quality, diversity, recognizability)
- At epoch 0 the Generator outputs pure noise. Within the first few epochs, blob-like stroke patterns appear, and by later epochs clear, recognizable handwritten digits are produced.
- The loss curves do not fall steadily as in normal supervised training; they oscillate because the Generator and Discriminator keep improving against each other. The Discriminator accuracy staying near 50–70% indicates a healthy adversarial balance.
- The PCA plot shows how closely generated-image features overlap real-image features; more overlap means more realistic output.
- The class-diversity bar chart shows whether all 10 digits are generated (no mode collapse) and the interpolation strip shows the latent space is smooth and continuous.
- Some generated digits may still be slightly distorted; longer training or more capacity improves this.

## RESULT
A Deep Convolutional GAN was implemented and trained on the MNIST dataset. The Generator successfully learned to produce realistic synthetic handwritten-digit images from random noise through adversarial training against the Discriminator, and the results were evaluated through visual inspection, loss curves, feature-space (PCA) comparison and a diversity check.

---
### Upload the notebook to GitHub (experiment management)
1. In Colab: **File → Save a copy in GitHub** (or **File → Download .ipynb** and upload manually).
2. Create/choose a repository (e.g. `24BAD081-DeepLearning-Lab`) and commit as `Expt 8 - GAN`.
3. Add the repository link in your record.